In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torchvision import transforms
from torchvision.models import vgg13, VGG13_Weights
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [2]:
CHECKPOINT_DIR = Path.cwd() / "checkpoints"

class VGG13Encoder(torch.nn.Module):
    def __init__(self, num_blocks, weights=VGG13_Weights.DEFAULT):
        super().__init__()
        self.num_blocks = num_blocks
        feature_extractor = vgg13(weights=weights).features
        self.blocks = torch.nn.ModuleList()
        for idx in range(self.num_blocks):
            start_idx = idx * 5
            end_idx = start_idx + 4
            layers = list(feature_extractor.children())[start_idx:end_idx]
            self.blocks.append(torch.nn.Sequential(*layers))

    def forward(self, x):
        activations = []
        for idx, block in enumerate(self.blocks):
            x = block(x)
            activations.append(x)
            if idx < self.num_blocks - 1:
                x = F.max_pool2d(x, kernel_size=2, stride=2)
        return activations


class DecoderBlock(torch.nn.Module):
    def __init__(self, out_channels):
        super().__init__()
        self.upconv = torch.nn.Conv2d(out_channels * 2, out_channels, kernel_size=3, padding=1)
        self.conv1 = torch.nn.Conv2d(out_channels * 2, out_channels, kernel_size=3, padding=1)
        self.conv2 = torch.nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.relu = torch.nn.ReLU()

    def forward(self, down, left):
        x = F.interpolate(down, scale_factor=2, mode="nearest")
        x = self.upconv(x)
        x = torch.cat([left, x], dim=1)
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        return x


class UNet(torch.nn.Module):
    def __init__(self, num_classes=1, num_blocks=4):
        super().__init__()
        self.channels = [64, 128, 256, 512, 512]
        self.encoder = VGG13Encoder(num_blocks=num_blocks)
        self.decoder = torch.nn.ModuleList()
        for i in range(num_blocks - 2, -1, -1):
            self.decoder.append(DecoderBlock(out_channels=self.channels[i]))
        self.final = torch.nn.Conv2d(self.channels[0], num_classes, kernel_size=1)

    def forward(self, x):
        activations = self.encoder(x)
        x = activations[-1]
        for i, block in enumerate(self.decoder):
            left = activations[-(i + 2)]
            x = block(down=x, left=left)
        return self.final(x)


class LinkNetDecoderBlock(torch.nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.upconv = torch.nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.conv1 = torch.nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.conv2 = torch.nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.relu = torch.nn.ReLU()

    def forward(self, down, left):
        x = F.interpolate(down, scale_factor=2, mode="nearest")
        x = self.relu(self.upconv(x))
        x = x + left
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        return x


class LinkNet(torch.nn.Module):
    def __init__(self, num_classes=1, num_blocks=4):
        super().__init__()
        self.channels = [64, 128, 256, 512, 512]
        self.encoder = VGG13Encoder(num_blocks=num_blocks)
        self.decoder = torch.nn.ModuleList()
        for i in range(num_blocks - 2, -1, -1):
            self.decoder.append(
                LinkNetDecoderBlock(in_channels=self.channels[i + 1], out_channels=self.channels[i])
            )
        self.final = torch.nn.Conv2d(self.channels[0], num_classes, kernel_size=1)

    def forward(self, x):
        activations = self.encoder(x)
        x = activations[-1]
        for i, block in enumerate(self.decoder):
            left = activations[-(i + 2)]
            x = block(down=x, left=left)
        return self.final(x)


class RegularizedDecoderBlock(torch.nn.Module):
    def __init__(self, out_channels, dropout_p=0.2):
        super().__init__()
        self.upconv = torch.nn.Conv2d(out_channels * 2, out_channels, kernel_size=3, padding=1)
        self.bn0 = torch.nn.BatchNorm2d(out_channels)
        self.conv1 = torch.nn.Conv2d(out_channels * 2, out_channels, kernel_size=3, padding=1)
        self.bn1 = torch.nn.BatchNorm2d(out_channels)
        self.conv2 = torch.nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = torch.nn.BatchNorm2d(out_channels)
        self.relu = torch.nn.ReLU()
        self.dropout = torch.nn.Dropout2d(p=dropout_p)

    def forward(self, down, left):
        x = F.interpolate(down, scale_factor=2, mode="nearest")
        x = self.relu(self.bn0(self.upconv(x)))
        x = torch.cat([left, x], dim=1)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.dropout(x)
        x = self.relu(self.bn2(self.conv2(x)))
        return x


class UNetRegularized(torch.nn.Module):
    def __init__(self, num_classes=1, num_blocks=4, dropout_p=0.2):
        super().__init__()
        self.channels = [64, 128, 256, 512, 512]
        self.encoder = VGG13Encoder(num_blocks=num_blocks)
        self.decoder = torch.nn.ModuleList()
        for i in range(num_blocks - 2, -1, -1):
            self.decoder.append(RegularizedDecoderBlock(out_channels=self.channels[i], dropout_p=dropout_p))
        self.final = torch.nn.Conv2d(self.channels[0], num_classes, kernel_size=1)

    def forward(self, x):
        activations = self.encoder(x)
        x = activations[-1]
        for i, block in enumerate(self.decoder):
            left = activations[-(i + 2)]
            x = block(down=x, left=left)
        return self.final(x)


MODELS = {
    "khoroshilova_unet_best": ("UNet", "khoroshilova_4._BCE(0.2)_+_Dice(0.8).pt"),
    "khoroshilova_linknet":   ("LinkNet", "khoroshilova_linknet.pt"),
    "safwan_best":            ("UNetRegularized", "safwan_best_unet_regularized.pt"),
}
print("Available models:", list(MODELS.keys()))

Available models: ['khoroshilova_unet_best', 'khoroshilova_linknet', 'safwan_best']


In [6]:
IMAGE_PATH = "demo_images/random_guy.png"   # yeah
MODEL_KEY  = "safwan_best"         
# ^ "khoroshilova_unet_best" | "khoroshilova_linknet" | "safwan_best" ^

In [ ]:
arch_name, ckpt_file = MODELS[MODEL_KEY]
ckpt_path = CHECKPOINT_DIR / ckpt_file
print(f"Model: {MODEL_KEY}  ({arch_name})")
print(f"Checkpoint: {ckpt_path}")

# Build model
model = {"UNet": UNet, "LinkNet": LinkNet, "UNetRegularized": UNetRegularized}[arch_name](
    num_classes=1, num_blocks=4
).to(device)

# Load weights
ckpt = torch.load(ckpt_path, map_location=device, weights_only=True)
if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
    model.load_state_dict(ckpt["model_state_dict"])
    print(f"Strategy: {ckpt.get('strategy')}  |  Saved IoU: {ckpt.get('iou', '?')}")
else:
    model.load_state_dict(ckpt)
model.eval()
print("Model loaded.\n")

# Load and preprocess image
img = Image.open(IMAGE_PATH).convert("RGB")
original_size = img.size  # (w, h)
img_resized = img.resize((240, 320))  # model expects 320h x 240w

normalize = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
input_tensor = normalize(img_resized).unsqueeze(0).to(device)

# Inference
with torch.no_grad():
    logits = model(input_tensor)
    prob = torch.sigmoid(logits).squeeze().cpu().numpy()
    mask = (prob > 0.5).astype("uint8")

print(f"Input image: {IMAGE_PATH}  (original {original_size[0]}x{original_size[1]}, resized to 240x320)")
print(f"Mask pixel coverage: {mask.sum()}/{mask.size} ({100 * mask.sum() / mask.size:.1f}%)")

# Display
fig, axes = plt.subplots(1, 3, figsize=(12, 5))

axes[0].imshow(img_resized)
axes[0].set_title("Input (resized)")
axes[0].axis("off")

axes[1].imshow(prob, cmap="hot", vmin=0, vmax=1)
axes[1].set_title("Probability map")
axes[1].axis("off")

axes[2].imshow(img_resized)
axes[2].imshow(mask, cmap="Greens", alpha=0.5)
axes[2].set_title("Segmentation overlay")
axes[2].axis("off")

fig.suptitle(f"{MODEL_KEY}  -  {IMAGE_PATH}", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()